# Retail Email Campaign Uplift Modeling

This notebook uses the randomized Hillstrom retail email experiment to estimate heterogeneous effects on **conversion**. The two email campaigns are modeled as separate treatments against the `No E-Mail` control. Only pre-treatment customer information is used as input.

**Scope warning:** these are retail-experiment estimates. They must not be applied directly to telecom customers; telecom targeting requires suitable telecom intervention data and its own validation.

In [ ]:
# Import the modeling, preprocessing, evaluation, and persistence tools used below.
from pathlib import Path
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from causalml.inference.meta import BaseTClassifier
from causalml.metrics import auuc_score, plot_gain, plot_qini, qini_score
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Locate the project root so paths work whether the kernel starts in the root or notebooks/.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "hillstrom.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate data/raw/hillstrom.csv from the current directory.")

DATA_PATH = ROOT / "data" / "raw" / "hillstrom.csv"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
RANDOM_STATE = 42
CONTROL_GROUP = "No E-Mail"

## Load and validate the experiment data

`segment` records randomized campaign assignment. The two email segments are treatment arms and `No E-Mail` is the control. `conversion` is the binary outcome; `visit` and `spend` occur after assignment and are intentionally excluded along with the assignment column.

In [ ]:
# Read the campaign experiment and fail clearly if its required columns are absent.
df = pd.read_csv(DATA_PATH)
required_columns = {
    "segment", "conversion", "visit", "spend", "recency", "history_segment",
    "history", "mens", "womens", "zip_code", "newbie", "channel",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Hillstrom data is missing required columns: {sorted(missing_columns)}")

# Confirm that only the expected randomized arms are present and the outcome is binary.
treatment_groups = sorted(df["segment"].dropna().unique().tolist())
if CONTROL_GROUP not in treatment_groups or len(treatment_groups) != 3:
    raise ValueError(f"Expected control {CONTROL_GROUP!r} and two email arms; found {treatment_groups}")
if df["conversion"].isna().any() or not set(df["conversion"].unique()).issubset({0, 1}):
    raise ValueError("conversion must be a non-null binary outcome coded as 0/1")

treatment_arms = [group for group in treatment_groups if group != CONTROL_GROUP]
print(f"Rows: {len(df):,}")
display(df.groupby("segment")["conversion"].agg(["count", "mean"]).rename(columns={"mean": "conversion_rate"}))
print("Treatment arms:", treatment_arms)
print("Control group:", CONTROL_GROUP)

## Define groups, outcome, and eligible features

The split is stratified by campaign assignment and outcome to preserve both treatment balance and conversion prevalence in the held-out sample. The feature list contains only customer information available before campaign assignment.

In [ ]:
# Keep only the experiment assignment, conversion outcome, and pre-treatment covariates.
feature_columns = [
    "recency", "history_segment", "history", "mens", "womens",
    "zip_code", "newbie", "channel",
]
excluded_columns = {"visit", "spend", "conversion", "segment"}
if excluded_columns.intersection(feature_columns):
    raise ValueError("Post-treatment, outcome, or treatment columns must not be model features")

strata = df["segment"].astype(str) + "_" + df["conversion"].astype(str)
train_index, test_index = train_test_split(
    df.index,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=strata,
)
train_data = df.loc[train_index].copy()
test_data = df.loc[test_index].copy()

X_train = train_data[feature_columns]
X_test = test_data[feature_columns]
y_train = train_data["conversion"].astype(int)
y_test = test_data["conversion"].astype(int)
treatment_train = train_data["segment"]
treatment_test = test_data["segment"]

print(f"Train rows: {len(train_data):,}; held-out test rows: {len(test_data):,}")
print("Training assignment counts:\n", treatment_train.value_counts().to_string())
print("Features (pre-treatment only):", feature_columns)

## Encode features without leaking test data

Categorical variables are one-hot encoded using categories learned from the training set only. Unseen test categories are handled safely, and numeric features pass through unchanged.

In [ ]:
# Fit preprocessing on training records only, then reuse it unchanged on the test set.
categorical_features = ["history_segment", "zip_code", "channel"]
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ],
    remainder="passthrough",
    sparse_threshold=0,
)
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)
print("Encoded train/test shapes:", X_train_encoded.shape, X_test_encoded.shape)

## Train the CausalML T-learner

`BaseTClassifier` fits separate LightGBM outcome models for each campaign arm and for control. The difference between each treatment and control conversion prediction is its estimated individual uplift.

In [ ]:
# Use a regularized LightGBM classifier as the shared learner for each arm's outcome model.
lightgbm_learner = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
uplift_model = BaseTClassifier(learner=lightgbm_learner, control_name=CONTROL_GROUP)
uplift_model.fit(
    X=X_train_encoded,
    treatment=treatment_train.to_numpy(),
    y=y_train.to_numpy(),
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# CausalML's prediction columns follow the treatment-group order stored by the fitted model.
model_treatment_arms = [str(group) for group in uplift_model.t_groups]
if set(model_treatment_arms) != set(treatment_arms):
    raise RuntimeError(f"Fitted model treatment arms do not match the data: {model_treatment_arms}")
print("Fitted treatment arms in prediction-column order:", model_treatment_arms)

## Predict held-out individual treatment effects

Predictions are made only after fitting and preprocessing are complete. Each predicted uplift is a difference in modeled conversion probability for an email arm versus the no-email control.

In [ ]:
# Generate one test-set uplift estimate per email campaign arm.
uplift_predictions = uplift_model.predict(X_test_encoded, verbose=False)
if uplift_predictions.shape != (len(test_data), len(model_treatment_arms)):
    raise RuntimeError(f"Unexpected uplift prediction shape: {uplift_predictions.shape}")

test_predictions = pd.DataFrame(
    {
        "treatment_group": treatment_test,
        "conversion": y_test,
    },
    index=test_data.index,
)
for column_index, treatment_group in enumerate(model_treatment_arms):
    safe_name = re.sub(r"[^a-z0-9]+", "_", treatment_group.lower()).strip("_")
    test_predictions[f"uplift_{safe_name}"] = uplift_predictions[:, column_index]

display(test_predictions.head())

## Held-out Qini and AUUC evaluation

For each email arm, the evaluation population includes only that arm and the shared control. This yields a binary treatment comparison for the CausalML Qini/AUUC metrics. Scores are based on observed held-out conversions, not the model's training outcomes.

In [ ]:
# Score each campaign against control separately using the arm-specific predicted uplift ranking.
metric_rows = []
evaluation_frames = {}
for column_index, treatment_group in enumerate(model_treatment_arms):
    arm_mask = treatment_test.isin([CONTROL_GROUP, treatment_group]).to_numpy()
    safe_name = re.sub(r"[^a-z0-9]+", "_", treatment_group.lower()).strip("_")
    evaluation_df = pd.DataFrame(
        {
            "y": y_test.to_numpy()[arm_mask],
            "w": (treatment_test.to_numpy()[arm_mask] == treatment_group).astype(int),
            "uplift_score": uplift_predictions[arm_mask, column_index],
        }
    )
    evaluation_frames[treatment_group] = evaluation_df

    # CausalML returns scores indexed by prediction-column name; normalize for comparability.
    qini = float(qini_score(evaluation_df, outcome_col="y", treatment_col="w", normalize=True)["uplift_score"])
    auuc = float(auuc_score(evaluation_df, outcome_col="y", treatment_col="w", normalize=True)["uplift_score"])
    metric_rows.append(
        {
            "treatment_group": treatment_group,
            "control_group": CONTROL_GROUP,
            "test_rows": len(evaluation_df),
            "treated_rows": int(evaluation_df["w"].sum()),
            "control_rows": int((evaluation_df["w"] == 0).sum()),
            "qini": qini,
            "auuc": auuc,
        }
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

## Evaluation plots

The Qini curve compares cumulative incremental conversions under uplift-based targeting with random targeting. The gain (AUUC) curve summarizes cumulative incremental conversion gains as more customers are targeted.

In [ ]:
# Plot both held-out curves for each arm, retaining the control comparison used for its score.
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for treatment_group, evaluation_df in evaluation_frames.items():
    plot_qini(
        evaluation_df,
        outcome_col="y",
        treatment_col="w",
        normalize=True,
        ax=axes[0],
    )
    plot_gain(
        evaluation_df,
        outcome_col="y",
        treatment_col="w",
        normalize=True,
        ax=axes[1],
    )

axes[0].set_title("Held-out Qini curves")
axes[1].set_title("Held-out cumulative gain (AUUC) curves")
axes[0].legend(model_treatment_arms, loc="best")
axes[1].legend(model_treatment_arms, loc="best")
fig.tight_layout()
REPORT_DIR.mkdir(parents=True, exist_ok=True)
evaluation_plot_path = REPORT_DIR / "uplift_evaluation_curves.png"
fig.savefig(evaluation_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved evaluation plots to {evaluation_plot_path}")

## Save the fitted model and evaluation results

The serialized bundle includes the fitted encoder and feature/treatment metadata required to reproduce predictions. Metrics and held-out predictions are stored separately for review.

In [ ]:
# Persist the model together with preprocessing metadata, held-out scores, and individual predictions.
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "uplift_t_learner.pkl"
metrics_path = REPORT_DIR / "uplift_model_metrics.csv"
predictions_path = REPORT_DIR / "uplift_test_predictions.csv"

joblib.dump(
    {
        "model": uplift_model,
        "preprocessor": preprocessor,
        "feature_columns": feature_columns,
        "control_group": CONTROL_GROUP,
        "treatment_groups": model_treatment_arms,
    },
    model_path,
)
metrics_df.to_csv(metrics_path, index=False)
test_predictions.to_csv(predictions_path, index_label="row_index")

print(f"Saved model bundle: {model_path}")
print(f"Saved held-out metrics: {metrics_path}")
print(f"Saved held-out predictions: {predictions_path}")
print(f"Saved evaluation plot: {evaluation_plot_path}")

## Interpretation boundary

Qini and AUUC summarize how well this model ranks incremental conversion response on held-out records from the Hillstrom retail experiment. They do not prove that the uplift estimates transport to another population or intervention. **Do not use these retail estimates directly for telecom customer decisions.** A telecom application requires relevant telecom treatment/outcome data, an appropriate experiment or identification strategy, and separate out-of-sample validation.